# 🏗️ YOLOv11 Training Notebook - Asbestos Detection

This notebook implements the pipeline to train a YOLOv11 model on your custom asbestos dataset.

### Prerequisites
* Ensure your `data.yaml` file is configured correctly.
* Ensure your dataset images are organized in `train`, `valid`, and `test` folders.

## 1. Install Ultralytics
We install the official library which includes YOLOv8 and YOLOv11.

In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries & Setup
Load the necessary modules and check for GPU availability.

In [2]:
from ultralytics import YOLO
import os
import torch

# Check if GPU is available
print(f"Setup complete. Using GPU: {torch.cuda.is_available()}")

Setup complete. Using GPU: True


## 3. Define Dataset Path
**ACTION REQUIRED:** Update the `data_yaml_path` variable below to point to your specific `data.yaml` file.

In [3]:
# REPLACE THIS PATH with the actual path to your data.yaml file
# Example: "C:/Users/You/Downloads/Asbestos_Dataset/data.yaml"

data_yaml_path = "C:/Users/aarya/Downloads/asbestos detection/asbestos.v2-release.yolov8/data.yaml"

if not os.path.exists(data_yaml_path):
    print(f"⚠️ WARNING: File not found at {data_yaml_path}. Please check the path.")
else:
    print(f"✅ Dataset found at {data_yaml_path}")

✅ Dataset found at C:/Users/aarya/Downloads/asbestos detection/asbestos.v2-release.yolov8/data.yaml


## 4. Load and Train Model
We load the **YOLOv11 Nano** (`yolo11n.pt`) model for a balance of speed and accuracy. 

* `epochs`: Number of training cycles (100 is a good start).
* `imgsz`: Image resolution (640 is standard).
* `batch`: Adjust this if you run out of memory (try 8 or 16).

In [6]:
# Load the pre-trained YOLOv11 model
model = YOLO("yolo11n.pt") 
results = model.train(
        data=data_yaml_path,
        
        # 1. QUALITY (Keep High Res)
        imgsz=1024,
        
        # 2. SPEED BOOSTERS
        batch=4,              # Keep low to prevent memory crashes
        workers=4,            # Uses 4 CPU cores to load data (Set to 0 if on Windows and it crashes)
        cache=True,           # RAM caching (huge speedup for small datasets)
        amp=True,             # Automatic Mixed Precision (Faster math)
        
        # 3. STRATEGY
        epochs=300,
        patience=50,
        close_mosaic=20,
        
        name='yolov11s_fast_highres'
    )

New https://pypi.org/project/ultralytics/8.4.25 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.229  Python-3.13.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/aarya/Downloads/asbestos detection/asbestos.v2-release.yolov8/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937,

## 5. Validate the Model
Evaluate the model's performance on the validation/test set to check for accuracy (mAP).

In [9]:
# 1. Define the path to your successful run
# Look at your file explorer: "runs" -> "detect" -> "yolov8_asbestos4" -> "weights" -> "best.pt"
model_path = "C:/Users/aarya/Downloads/asbestos detection/asbestos.v2-release.yolov8/runs/detect/yolov11s_fast_highres/weights/best.pt" 

# 2. Load that specific model
model = YOLO(model_path)

# 3. Run validation
metrics = model.val(split='test')

Ultralytics 8.3.229  Python-3.13.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2.50.3 MB/s, size: 46.8 KB)
val: Scanning C:\Users\aarya\Downloads\asbestos detection\asbestos.v2-release.yolov8\test\labels.cache... 133 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 133/133 85.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.2it/s 7.6s0.6ss
                   all        133       1117      0.745      0.642      0.677      0.496
       thick-dark-mark         30        207      0.894      0.855      0.913      0.697
      thick-light-mark        130        671      0.753      0.645      0.702      0.467
        thin-dark-mark         53        150      0.861      0.662      0.707      0.541
       thin-light-mark         53         89      0.471      0.4

In [10]:
# 3. Extract the Metrics
# YOLOv8 stores these inside the 'metrics.box' object
precision = metrics.box.mp   # Mean Precision
recall = metrics.box.mr      # Mean Recall
map50 = metrics.box.map50    # mAP@50 (Often used as "Accuracy" for object detection)
map50_95 = metrics.box.map   # mAP@50-95 (Stricter accuracy)

# 4. Calculate F1 Score
# Formula: F1 = 2 * (Precision * Recall) / (Precision + Recall)
# We add a small epsilon (1e-7) to avoid division by zero errors
f1_score = 2 * (precision * recall) / (precision + recall + 1e-7)

# 5. Print the Report
print("\n" + "="*40)
print("       🏆 MODEL PERFORMANCE REPORT       ")
print("="*40)
print(f"✅ Precision:   {precision:.4f}  (How confident the model is)")
print(f"✅ Recall:      {recall:.4f}     (How many fibers it found)")
print(f"✅ F1 Score:    {f1_score:.4f}   (Balance between P and R)")
print("-" * 40)
print(f"🎯 Accuracy (mAP@50):    {map50:.4f}")
print(f"🎯 Strict Acc (mAP50-95): {map50_95:.4f}")
print("="*40)


       🏆 MODEL PERFORMANCE REPORT       
✅ Precision:   0.7449  (How confident the model is)
✅ Recall:      0.6417     (How many fibers it found)
✅ F1 Score:    0.6894   (Balance between P and R)
----------------------------------------
🎯 Accuracy (mAP@50):    0.6772
🎯 Strict Acc (mAP50-95): 0.4958


## 6. Run Inference (Test Prediction)
Test the trained model on a specific image to see the bounding boxes.

In [ ]:
# REPLACE with a path to an image from your test set
test_image = "C:/Users/aarya/Downloads/asbestos detection/asbestos.v2-release.yolov8/test/images/pc450newF-091_jpg.rf.9daf42a591c6d9a3cc8f0ff3a955b482.jpg"

if os.path.exists(test_image):
    results = model.predict(
        source=test_image,
        save=True,
        conf=0.5  # Confidence threshold
    )
    print("Prediction saved!")
else:
    print("Test image not found. Please update the path.")







image 1/1 C:\Users\aarya\Downloads\asbestos detection\asbestos.v2-release.yolov8\test\images\pc450BP0055_synt2_jpg.rf.f24b89373413201610a6e580b22935c3.jpg: 1024x1024 4 thick-dark-marks, 2 thick-light-marks, 41.5ms
Speed: 2546.6ms preprocess, 41.5ms inference, 9.7ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to C:\Users\aarya\Downloads\asbestos detection\asbestos.v2-release.yolov8\runs\detect\predict7
Prediction saved!
